# Notebook 43 - Does CIC-IoMT-2024 collapse when it is recovered *without* clipping?

**Why this notebook.** Notebook 34 ran the CICIoT2023 recovery recipe on CIC-IoMT-2024 (320 dual-mode evaluations) and found no normalisation collapse; the one preprocessing difference that bears on the mechanism is that the IoMT pipeline clips every feature to its 0.1 and 99.9 training percentiles. Notebook 35 (T1) showed that clipping removes every CICIoT2023 collapse, and notebook 42 showed that the unclipped IoMT training rows carry the same kind of trigger (255 rows with a standardised value above 100, the largest 598). What is still missing is the direct test: recover IoMT detectors on the **unclipped** rows under the same recipe and watch the BatchNorm running variance batch by batch. If a collapse appears, the paper's second-corpus sentence becomes an observation instead of a prediction. If none appears, the per-batch trace says whether the extreme rows were too small to matter or simply never landed late in a unit.

**What it does.** GPU. (1) Loads, deduplicates and splits the IoMT training rows exactly as `src/saber/bridge_iomt.py` (same loader, same rule, same seeded carve-out; the official test files are never opened) and standardises them with their own unclipped mean and standard deviation, no clipping; the tensors are cached under `data/` (git-ignored) with a committed manifest. (2) Trains a shallow teacher on the unclipped rows with notebook 32's recipe. (3) Recovers the dense teacher and its Fisher-pruned student (notebook 33's frozen 40% structure) on seeded 10% subsets for eight units, five seeds each, with the dual-mode audit at every unit and a per-batch trace (largest |z| in the batch, largest BatchNorm running variance after the step, stored-statistics benign escalation every 25 batches). (4) Recalibrates every collapse snapshot on fixed held-out batches. (5) Writes the verdict under the readings fixed in stage 2.

**Readings, fixed in stage 2 before any data is opened.** V1 holds if at least one of the 80 unit evaluations is a normalisation collapse under the IoMT gap rule of notebook 34 (stored-statistics benign escalation exceeds the batch-statistics escalation by more than 0.10). The verdict is written either way, with the trace analysis that says why.

In [ ]:
# Stage 1 - bootstrap
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, gc, time, copy, hashlib, subprocess, importlib
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt

REPO = Path("/content/drive/MyDrive/SABER_Research/saber-ids")
IOMT_REPO = Path("/content/drive/MyDrive/IOMT_Compression_Research/iomt-compression-research")
os.chdir(REPO)
for p in (str(REPO), str(IOMT_REPO / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()
import src.saber.bridge_iomt as BI
from src.saber.bridge_iomt import iomt_taxonomy
from src.saber.taxonomy import DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit
from src.saber.surgery import prune_cnn1d_channels

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"
OUT = R / "43_iomt_unclipped_recovery"; OUT.mkdir(parents=True, exist_ok=True)
TRACE_DIR = OUT / "traces"; TRACE_DIR.mkdir(exist_ok=True)
SNAP_DIR = OUT / "collapse_snapshots"; SNAP_DIR.mkdir(exist_ok=True)          # .pt files are git-ignored
DATA_CACHE = REPO / "data/iomt_bridge_unclipped"; DATA_CACHE.mkdir(parents=True, exist_ok=True)   # git-ignored tensors
MODEL_DIR = REPO / "models/iomt_unclipped"; MODEL_DIR.mkdir(parents=True, exist_ok=True)            # git-ignored
BRIDGE_MANIFEST = json.loads((R / "32_iomt_bridge/bridge_manifest.json").read_text())           # the clipped bridge, committed
STRUCTURE_CSV = R / "33_iomt_scores_structures/shallow_fisher_r40_removed_groups.csv"
NB35_TRACE = R / "35_trigger_isolation/baseline_trace_shallow_fisher_s307.csv"
NB42_VERDICT = R / "42_iomt_extreme_values/U42_verdict.json"
assert STRUCTURE_CSV.exists() and NB35_TRACE.exists() and NB42_VERDICT.exists(), "upstream result files missing; pull the repo"
print("device:", DEVICE, "| torch", torch.__version__, "| bridge n_train", BRIDGE_MANIFEST["n_train"], "| features", BRIDGE_MANIFEST["n_features"])

In [ ]:
# Stage 2 - pre-registration (frozen: rerunning this cell can never change it after its commit)
PREREG = {
    "notebook": "43_iomt_unclipped_recovery",
    "question": ("does a shallow IoMT detector, recovered on CIC-IoMT-2024 training rows standardised WITHOUT clipping under the "
                 "CICIoT2023 recipe, show the BatchNorm normalisation collapse that the clipped IoMT pipeline never showed (NB34)"),
    "motivation": ("NB34: 320 dual-mode evaluations on the clipped IoMT pipeline, no collapse. NB35 T1: clipping removes every CICIoT2023 "
                   "collapse. NB42: the unclipped IoMT training rows carry 255 rows with |z| above 100 (largest 598.3, one-row batch-variance "
                   "bound 350). The paper currently states the unclipped collapse as predicted, not observed; this notebook observes it or not"),
    "design": {
        "corpus": ("CIC-IoMT-2024 WiFi/MQTT, official training files only, exact duplicates removed, seeded 20% validation carve-out, "
                   "exactly as src/saber/bridge_iomt.build_bridge and NB42; the official test files are never opened"),
        "preprocessing": ("UNCLIPPED: features standardised with the mean and standard deviation of the unclipped training rows; "
                          "no clipping anywhere; validation rows standardised with the same training statistics"),
        "reproduction_checks": ("training rows must equal the bridge manifest n_train (3,612,063), class names must equal the bridge "
                                "manifest class list, and the largest unclipped |z| must agree with NB42 (598.3) or the notebook stops"),
        "teacher": ("shallow CNN1D (Conv 64, BN, Conv 128, BN, GAP, linear) trained on the unclipped rows with NB32's recipe: 4 full epochs, "
                    "Adam 1e-3, batch 1,024, cross-entropy with class weights count^-0.5 (the clipped shallow teacher's exponent), seed 0; "
                    "its dual-mode audit is recorded; if its own stored-over-batch gap exceeds 0.10 it is recalibrated before use and that "
                    "is recorded; NB32's acceptance rule (b2a <= 0.10, attack recall >= 0.90, fine macro-F1 >= 0.50) is reported, not required"),
        "cells": ("dense (the teacher itself) and fisher-pruned (NB33 frozen shallow Fisher 40% structure applied to the unclipped teacher, "
                  "minimum 8 channels per layer) x seeds [101, 211, 307, 401, 503]; recovery: seeded 10% subset of the training rows, "
                  "batches of 1,024, Adam 1e-3, the same class weights, BatchNorm momentum 0.1, 8 units; identical to NB31/NB34/NB35"),
        "audit": ("at the end of every unit: stored-statistics evaluation and batch-statistics probe (training mode, batches of 8,192, "
                  "momentum 0, running statistics restored) on the class-capped validation subsample (at most 4,000 rows per class, seed 0)"),
        "trace": ("every batch of every run: largest |z| in the batch and largest BatchNorm running variance after the step; "
                  "stored-statistics benign escalation every 25 batches"),
        "collapse_rule": ("IoMT gap rule of NB34: eval-mode benign escalation minus batch-statistics benign escalation > 0.10; the CICIoT2023 "
                          "absolute rule (eval > 0.10 and batch <= 0.10) is recorded alongside for reference"),
        "remedy": ("every collapse snapshot is recalibrated with no gradient step on 50 batches of 1,024 training rows from one fixed slice "
                   "(seed 2026, as NB34) and re-evaluated"),
        "late_landing": ("the batch holding the largest |z| of a unit arrives within the last 25 batches of that unit (the decay window of "
                         "equation (4) of the paper for momentum 0.1)"),
    },
    "claims": {
        "V1": "at least one normalisation collapse (gap rule) in the 80 unit evaluations (2 cells x 5 seeds x 8 units)",
        "V2": ("descriptive: per unit, the largest single-batch jump in the maximum running variance occurs at the batch holding the "
               "largest |z| of that unit; the jump size is reported beside the CICIoT2023 trace of NB35 (jumps 2.27 to 2.45 at |z| 1,390)"),
        "V3": "if V1 holds: recalibration brings every collapse snapshot's eval-mode benign escalation to within 0.02 of its batch-statistics value",
    },
    "readings": {
        "V1_held": "unclipped_iomt_collapses: the paper's second-corpus sentence becomes an observation; the generality claim stands",
        "V1_not_held_with_late_landing": ("spike_too_small: the IoMT extreme rows inflate the variance but not enough to collapse at |z| 598; "
                                          "the paper must say so and tie the failure to the size of the extreme value"),
        "V1_not_held_without_late_landing": "untested_by_order: no extreme batch landed late in any unit; add order seeds before concluding",
    },
    "reporting_rule": "V1 and V3 reported as held or not held, V2 described; every unit row and every trace written either way",
    "seeds": [101, 211, 307, 401, 503], "units": 8, "subset_fraction": 0.10, "batch_size": 1024, "lr": 1e-3,
    "class_weight_exponent": 0.5, "bn_momentum": 0.1, "min_channels_per_layer": 8, "collapse_gap": 0.10, "cal_seed": 2026,
    "no_test_access": True, "no_new_selection": True,
}
pre_path = OUT / "V43_PREREGISTRATION.json"
if pre_path.exists():
    assert json.loads(pre_path.read_text()) == PREREG, "pre-registration on disk differs from this cell; do not edit a committed pre-registration"
    print("pre-registration already on disk and identical")
else:
    pre_path.write_text(json.dumps(PREREG, indent=2)); print("pre-registration written:", pre_path)

SEEDS = PREREG["seeds"]; N_UNITS = PREREG["units"]; SUBSET_FRACTION = PREREG["subset_fraction"]; BATCH = PREREG["batch_size"]
LR = PREREG["lr"]; ALPHA = PREREG["class_weight_exponent"]; MIN_W = PREREG["min_channels_per_layer"]
COLLAPSE_GAP = PREREG["collapse_gap"]; CAL_SEED = PREREG["cal_seed"]; LATE_WINDOW = 25; TRACE_EVERY = 25

In [ ]:
# Stage 2b - commit the pre-registration before any result exists (skipped automatically once results exist)
def _prereg_committed():
    rel = "results/saber/43_iomt_unclipped_recovery/V43_PREREGISTRATION.json"
    h = subprocess.run(["git", "log", "-1", "--format=%h", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    dirty = subprocess.run(["git", "status", "--porcelain", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    return bool(h) and not dirty


_results_exist = any(p.is_file() and p.name != "V43_PREREGISTRATION.json" for p in OUT.rglob("*"))
if _results_exist:
    assert _prereg_committed(), ("results exist but the pre-registration is not committed, so it can no longer be committed before any "
                                 "result; stop here and report it")
    print("results already exist in", OUT.name, "and the pre-registration is committed; nothing to do here")
else:
    os.chdir(REPO)
    r = subprocess.run(["python", "tools/commit_cell.py",
                        "NB43 pre-registration, committed before any result: does CIC-IoMT-2024 collapse when recovered WITHOUT clipping? "
                        "Unclipped standardisation of the bridge's own training rows (reproduction checks against the bridge manifest and NB42), "
                        "shallow teacher trained with the NB32 recipe, dense and Fisher-pruned (NB33 structure) cells x 5 seeds x 8 units under "
                        "the NB31/34/35 recovery recipe, dual-mode audit every unit, per-batch trace of |z| and running variance, "
                        "recalibration of every collapse snapshot. V1 = at least one gap-rule collapse in 80 unit evaluations; readings "
                        "unclipped_iomt_collapses / spike_too_small / untested_by_order. No test access."], capture_output=True, text=True)
    print(r.stdout); print(r.stderr)

In [ ]:
# Stage 3 - model, audit, recovery and trace helpers, with a self-test on synthetic tensors before any real row is opened
class CNN1D(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(1, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(64),
                                  nn.Conv1d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(128))
        self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(128, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


def bn_modules(model):
    return [(n, m) for n, m in model.named_modules() if isinstance(m, nn.BatchNorm1d) and m.running_mean is not None]


def bn_var_max(model):
    return float(max(m.running_var.max().item() for _, m in bn_modules(model)))


def set_bn_momentum(model, momentum):
    for _, m in bn_modules(model):
        m.momentum = momentum


def forward_logits(model, X):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()


def forward_logits_batch_stats(model, X):
    # training-mode probe with momentum 0: the stored running statistics are read by nobody and changed by nothing
    saved = {n: (m.running_mean.clone(), m.running_var.clone(), m.momentum, m.num_batches_tracked.clone()) for n, m in bn_modules(model)}
    model.train()
    for n, m in bn_modules(model):
        m.momentum = 0.0
    with torch.no_grad():
        out = torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()
    for n, m in bn_modules(model):
        rm, rv, mom, nbt = saved[n]
        m.running_mean.copy_(rm); m.running_var.copy_(rv); m.momentum = mom; m.num_batches_tracked.copy_(nbt)
    model.eval()
    return out


def audit_from_logits(lg, y, taxonomy):
    a = full_model_audit(lg, y, taxonomy, DEFAULT_COST_PROFILES)
    return {"b2a": float(a["benign_to_attack_rate"]), "a2b": float(a["attack_to_benign_rate"]), "attack_recall": float(a["binary_attack_recall"]),
            "family_f1": float(a["family_macro_f1"]), "fine_f1": float(a["fine_macro_f1"]), "ece": float(a["ece15"])}


def audit(model, X, y, taxonomy, mode="eval"):
    lg = forward_logits(model, X) if mode == "eval" else forward_logits_batch_stats(model, X)
    return audit_from_logits(lg, y, taxonomy)


def b2a_only(model, X, y, taxonomy):
    # stored-statistics benign escalation alone (for the per-batch trace): share of benign rows not predicted benign
    b = taxonomy.benign_index; pred = forward_logits(model, X).argmax(1)
    return float((pred[y == b] != b).mean())


def class_weights(counts, alpha):
    w = np.zeros(len(counts), dtype=np.float64); nz = counts > 0
    w[nz] = 1.0 / np.power(counts[nz], alpha); w[nz] /= w[nz].mean()
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


def make_subset_loader(dataset, seed, fraction, batch):
    n = len(dataset)
    sub = torch.randperm(n, generator=torch.Generator().manual_seed(seed))[: int(n * fraction)]
    return torch.utils.data.DataLoader(torch.utils.data.Subset(dataset, sub.tolist()), batch_size=batch, shuffle=True,
                                       generator=torch.Generator().manual_seed(seed)), sub


def recalibrate_bn(model, cal_loader, n_batches=50):
    for _, m in bn_modules(model):
        m.reset_running_stats(); m.momentum = None           # cumulative averaging
    model.train()
    with torch.no_grad():
        for i, (xb, _) in enumerate(cal_loader):
            if i >= n_batches:
                break
            model(xb.to(DEVICE))
    set_bn_momentum(model, 0.1); model.eval()
    return model


def recover(model, loader, n_units, weights, X_eval, y_eval, taxonomy, tag, trace_every=25, gap=0.10, on_collapse=None):
    # Recovers `model` in place under the paper's recipe; returns per-unit rows and per-batch trace rows.
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    lossf = nn.CrossEntropyLoss(weight=weights)
    set_bn_momentum(model, 0.1)
    rows, trace = [], []
    for unit in range(1, n_units + 1):
        for bi, (xb, yb) in enumerate(loader):
            model.train(); xb = xb.to(DEVICE)
            opt.zero_grad(); lossf(model(xb), yb.to(DEVICE)).backward(); opt.step()
            tr = {"tag": tag, "unit": unit, "batch": bi, "batch_max_abs_z": float(xb.abs().max()), "bn_var_max": bn_var_max(model)}
            if bi % trace_every == 0:
                tr["eval_b2a"] = b2a_only(model, X_eval, y_eval, taxonomy)
            trace.append(tr)
        ev = audit(model, X_eval, y_eval, taxonomy, "eval"); bs = audit(model, X_eval, y_eval, taxonomy, "batch")
        col_gap = bool(ev["b2a"] - bs["b2a"] > gap); col_abs = bool(ev["b2a"] > 0.10 and bs["b2a"] <= 0.10)
        row = {"tag": tag, "unit": unit, "n_batches": bi + 1, **{f"eval_{k}": v for k, v in ev.items()}, **{f"batch_{k}": v for k, v in bs.items()},
               "gap": ev["b2a"] - bs["b2a"], "is_collapse_gap": col_gap, "is_collapse_abs": col_abs, "bn_var_max": bn_var_max(model)}
        if col_gap and on_collapse is not None:
            row.update(on_collapse(model, unit))
        rows.append(row)
    model.eval()
    return rows, trace


def trace_analysis(tr, late_window=25):
    # Per unit: the batch of largest |z|, the running-variance jump at that batch, its rank among single-batch jumps, and whether it landed late.
    out = []
    for unit, g in tr.groupby("unit"):
        g = g.sort_values("batch"); v = g.bn_var_max.to_numpy(); z = g.batch_max_abs_z.to_numpy(); nb = len(g)
        jump = np.diff(np.r_[v[0], v]); iz = int(np.argmax(z)); order = np.argsort(-jump)
        out.append({"unit": int(unit), "n_batches": nb, "max_abs_z": float(z[iz]), "max_abs_z_batch": int(iz),
                    "jump_at_max_abs_z_batch": float(jump[iz]), "jump_rank": int(np.where(order == iz)[0][0]) + 1,
                    "largest_jump": float(jump.max()), "var_before": float(v[iz - 1]) if iz > 0 else float("nan"), "var_after": float(v[iz]),
                    "final_var": float(v[-1]), "late_landing": bool(iz >= nb - late_window)})
    return out


# ---- self-test on synthetic tensors (CPU or GPU): shapes, probe invariance, trace bookkeeping, planted extreme row found ----
_g = torch.Generator().manual_seed(0)
_Xs = torch.randn(6000, 45, generator=_g); _ys = torch.randint(0, 19, (6000,), generator=_g)
_Xs[4321, 7] = 500.0                                                           # one planted extreme value
_tax = iomt_taxonomy(BRIDGE_MANIFEST["class_names"], BRIDGE_MANIFEST["family_by_class"])
_m = CNN1D(19).to(DEVICE); _Xe = _Xs[:1500].to(DEVICE); _ye = _ys[:1500].numpy()
_b = {n: (m.running_mean.clone(), m.running_var.clone()) for n, m in bn_modules(_m)}
_e1 = forward_logits(_m, _Xe); _p = forward_logits_batch_stats(_m, _Xe); _e2 = forward_logits(_m, _Xe)
for n, m in bn_modules(_m):
    assert torch.equal(m.running_mean, _b[n][0]) and torch.equal(m.running_var, _b[n][1]), "probe changed the running statistics"
assert np.allclose(_e1, _e2, atol=1e-5) and not _m.training
_ld, _sub = make_subset_loader(torch.utils.data.TensorDataset(_Xs, _ys), seed=1, fraction=1.0, batch=256)
_rows, _tr = recover(_m, _ld, 1, class_weights(np.bincount(_ys.numpy(), minlength=19), ALPHA), _Xe, _ye, _tax, tag="selftest", trace_every=5)
_tr = pd.DataFrame(_tr); _an = trace_analysis(_tr, late_window=3)[0]
assert abs(b2a_only(_m, _Xe, _ye, _tax) - audit(_m, _Xe, _ye, _tax, "eval")["b2a"]) < 1e-9, "trace escalation differs from the audit's"
assert len(_rows) == 1 and _rows[0]["n_batches"] == 24 and len(_tr) == 24 and _tr.eval_b2a.notna().sum() == 5
assert abs(_an["max_abs_z"] - 500.0) < 1e-3 and 0 <= _an["max_abs_z_batch"] < 24 and 1 <= _an["jump_rank"] <= 24
_w0 = {n: p.detach().clone() for n, p in _m.named_parameters()}
recalibrate_bn(_m, torch.utils.data.DataLoader(torch.utils.data.TensorDataset(_Xs, _ys), batch_size=256), n_batches=3)
assert all(torch.equal(p, _w0[n]) for n, p in _m.named_parameters()), "recalibration touched a weight"
del _m, _Xs, _ys, _Xe; gc.collect()
print("self-test passed: probe non-destructive, one unit of", _rows[0]["n_batches"], "batches traced, planted |z| 500 found in batch",
      _an["max_abs_z_batch"], "with jump rank", _an["jump_rank"], "| recalibration leaves weights untouched")

In [ ]:
# Stage 4 - the UNCLIPPED IoMT tensors: load, deduplicate and split exactly as the bridge (cached after the first run)
TENSORS = DATA_CACHE / "tensors.pt"; UMAN = DATA_CACHE / "manifest_unclipped.json"
if TENSORS.exists() and UMAN.exists():
    UMANIFEST = json.loads(UMAN.read_text()); blob = torch.load(TENSORS, map_location="cpu", weights_only=False)
    print("unclipped cache loaded:", {k: tuple(v.shape) for k, v in blob.items()})
else:
    t0 = time.time()
    if BI.IOMT_SRC not in sys.path:
        sys.path.insert(0, BI.IOMT_SRC)
    import iomtc_data as D                                   # the IoMT project's loader, used as-is (same as the bridge and NB42)
    train_df, _test_unused = D.load_official(add_targets=True)
    del _test_unused; gc.collect()                           # the official test files are never touched
    feat_cols = [c for c in D.feature_columns(train_df)]
    drop, leak_note = BI._leakage_drop_list()
    feat_cols = [c for c in feat_cols if c not in set(drop)]
    assert feat_cols == BRIDGE_MANIFEST["feature_columns"], "feature columns differ from the committed bridge manifest; stop"
    n_raw = int(len(train_df))
    train_df = train_df.drop_duplicates(subset=feat_cols + ["attack_type"]).reset_index(drop=True)
    n_dedup = int(len(train_df))
    assert n_raw == BRIDGE_MANIFEST["n_official_train_rows"] and n_dedup == BRIDGE_MANIFEST["n_after_exact_dedup"], (n_raw, n_dedup)
    rng = np.random.default_rng(BI.SEED)
    val_mask = np.zeros(n_dedup, dtype=bool)
    for _, idx in train_df.groupby("attack_type").indices.items():
        idx = np.asarray(idx); rng.shuffle(idx)
        val_mask[idx[: int(round(BI.VAL_FRACTION * len(idx)))]] = True
    class_names = sorted(train_df["attack_type"].astype(str).unique().tolist())
    assert class_names == BRIDGE_MANIFEST["class_names"], "class list differs from the committed bridge manifest; stop"
    cls_index = {c: i for i, c in enumerate(class_names)}
    y_all = train_df["attack_type"].astype(str).map(cls_index).to_numpy(np.int64)
    X_tr = train_df.loc[~val_mask, feat_cols].to_numpy(np.float32); X_va = train_df.loc[val_mask, feat_cols].to_numpy(np.float32)
    y_tr, y_va = y_all[~val_mask], y_all[val_mask]
    del train_df; gc.collect()
    assert X_tr.shape[0] == BRIDGE_MANIFEST["n_train"] and X_va.shape[0] == BRIDGE_MANIFEST["n_val"], (X_tr.shape, X_va.shape)
    mean = X_tr.mean(axis=0, dtype=np.float64); std = X_tr.std(axis=0, dtype=np.float64); std = np.where(std < 1e-8, 1.0, std)
    for s_ in range(0, X_tr.shape[0], 250_000):                                   # standardise in place, no clipping
        X_tr[s_:s_ + 250_000] = ((X_tr[s_:s_ + 250_000] - mean) / std).astype(np.float32)
    for s_ in range(0, X_va.shape[0], 250_000):
        X_va[s_:s_ + 250_000] = ((X_va[s_:s_ + 250_000] - mean) / std).astype(np.float32)
    max_abs_z = float(np.abs(X_tr).max())
    blob = {"X_tr": torch.from_numpy(X_tr), "y_tr": torch.from_numpy(y_tr), "X_va": torch.from_numpy(X_va), "y_va": torch.from_numpy(y_va)}
    torch.save(blob, TENSORS)
    UMANIFEST = {"dataset": "CIC-IoMT-2024 WiFi/MQTT, official CIC train side; official test never opened; UNCLIPPED standardisation",
                 "n_official_train_rows": n_raw, "n_after_exact_dedup": n_dedup, "val_fraction": BI.VAL_FRACTION, "split_seed": BI.SEED,
                 "n_train": int(len(y_tr)), "n_val": int(len(y_va)), "feature_columns": feat_cols, "n_features": len(feat_cols),
                 "leakage_note": leak_note, "clipping": "none", "scaler_mean": mean.tolist(), "scaler_std": std.tolist(),
                 "class_names": class_names, "family_by_class": BRIDGE_MANIFEST["family_by_class"],
                 "train_class_counts": np.bincount(y_tr, minlength=len(class_names)).tolist(),
                 "max_abs_z_train": max_abs_z, "max_abs_z_val": float(np.abs(X_va).max()),
                 "sha256_X_tr": BI._sha256_array(X_tr), "sha256_y_tr": BI._sha256_array(y_tr),
                 "sha256_X_va": BI._sha256_array(X_va), "sha256_y_va": BI._sha256_array(y_va)}
    UMAN.write_text(json.dumps(UMANIFEST, indent=2)); del X_tr, X_va; gc.collect()
    print("unclipped tensors built and cached in", round(time.time() - t0, 1), "s")

nb42 = json.loads(NB42_VERDICT.read_text())["unclipped_iomt"]
assert abs(UMANIFEST["max_abs_z_train"] - nb42["max_abs_z"]) < 0.5, (UMANIFEST["max_abs_z_train"], nb42["max_abs_z"])   # reproduction check
(OUT / "manifest_unclipped.json").write_text(json.dumps({k: v for k, v in UMANIFEST.items() if k not in ("scaler_mean", "scaler_std")}, indent=2))

CLASS_NAMES = UMANIFEST["class_names"]; N_CLASSES = len(CLASS_NAMES)
taxonomy = iomt_taxonomy(CLASS_NAMES, UMANIFEST["family_by_class"])
TRAIN_DS = torch.utils.data.TensorDataset(blob["X_tr"], blob["y_tr"]); N_TRAIN = len(TRAIN_DS)
COUNTS = np.bincount(blob["y_tr"].numpy(), minlength=N_CLASSES); CLASS_W = class_weights(COUNTS, ALPHA)
Xv, yv_all = blob["X_va"], blob["y_va"].numpy()
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(yv_all == c)[0])[:4000] for c in range(N_CLASSES) if (yv_all == c).sum() > 0])
_idx = np.random.default_rng(12345).permutation(_idx)
EX_X = Xv[_idx].to(DEVICE); EX_Y = yv_all[_idx]; EXAMPLE_INPUT = Xv[:8].float().to(DEVICE)
_cal_idx = torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(CAL_SEED))[:50 * BATCH]
CAL_LOADER = torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_DS, _cal_idx.tolist()), batch_size=BATCH, shuffle=False)
print("training rows", N_TRAIN, "| evaluation rows", len(_idx), "| benign share", round(float((EX_Y == taxonomy.benign_index).mean()), 4),
      f"| largest unclipped |z| in training rows {UMANIFEST['max_abs_z_train']:.1f} (NB42: {nb42['max_abs_z']:.1f})",
      f"| in validation rows {UMANIFEST['max_abs_z_val']:.1f}")

In [ ]:
# Stage 5 - the unclipped shallow teacher (NB32 recipe), cached; dual-mode audit recorded
TEACHER_PT = MODEL_DIR / "shallow_teacher_seed0.pt"; TAUD = OUT / "teacher_audit.json"
if TEACHER_PT.exists():
    payload = torch.load(TEACHER_PT, map_location="cpu", weights_only=False)
    TEACHER = CNN1D(N_CLASSES); TEACHER.load_state_dict(payload["state_dict"]); TEACHER = TEACHER.to(DEVICE).eval()
    print("teacher loaded from cache")
else:
    t0 = time.time(); torch.manual_seed(0); np.random.seed(0)
    TEACHER = CNN1D(N_CLASSES).to(DEVICE)
    loader = torch.utils.data.DataLoader(TRAIN_DS, batch_size=BATCH, shuffle=True, generator=torch.Generator().manual_seed(0))
    opt = torch.optim.Adam(TEACHER.parameters(), lr=LR); lossf = nn.CrossEntropyLoss(weight=CLASS_W)
    for ep in range(1, 5):
        TEACHER.train()
        for xb, yb in loader:
            opt.zero_grad(); lossf(TEACHER(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
        print(f"  epoch {ep}: eval b2a {audit(TEACHER, EX_X, EX_Y, taxonomy, 'eval')['b2a']:.4f}", flush=True)
    TEACHER.eval()
    torch.save({"state_dict": {k: v.detach().cpu().clone() for k, v in TEACHER.state_dict().items()}, "alpha": ALPHA, "epochs": 4, "seed": 0}, TEACHER_PT)
    print("teacher trained in", round(time.time() - t0, 1), "s")

t_eval = audit(TEACHER, EX_X, EX_Y, taxonomy, "eval"); t_batch = audit(TEACHER, EX_X, EX_Y, taxonomy, "batch")
teacher_record = {"recipe": "4 full epochs, Adam 1e-3, batch 1024, class weights count^-0.5, seed 0, UNCLIPPED inputs",
                  "as_trained_eval": t_eval, "as_trained_batch": t_batch, "as_trained_gap": t_eval["b2a"] - t_batch["b2a"],
                  "as_trained_bn_var_max": bn_var_max(TEACHER), "recalibrated_before_use": False,
                  "meets_nb32_rule": bool(t_eval["b2a"] <= 0.10 and t_eval["attack_recall"] >= 0.90 and t_eval["fine_f1"] >= 0.50),
                  "clipped_teacher_reference_nb32": {"b2a": 0.08725, "a2b": 0.018429, "fine_f1": 0.702181, "family_f1": 0.802586}}
if teacher_record["as_trained_gap"] > COLLAPSE_GAP:                 # pre-registered rule: a collapsed teacher is recalibrated before use
    recalibrate_bn(TEACHER, CAL_LOADER); teacher_record["recalibrated_before_use"] = True
    teacher_record["after_recalibration_eval"] = audit(TEACHER, EX_X, EX_Y, taxonomy, "eval")
    print("teacher's own stored-over-batch gap exceeded 0.10; recalibrated before use")
TEACHER_SHA = hashlib.sha256(TEACHER_PT.read_bytes()).hexdigest(); teacher_record["checkpoint_sha256"] = TEACHER_SHA
TAUD.write_text(json.dumps(teacher_record, indent=2))
print("unclipped teacher: eval", {k: round(v, 4) for k, v in t_eval.items()}, "| batch b2a", round(t_batch["b2a"], 4),
      "| meets NB32 rule:", teacher_record["meets_nb32_rule"], "| recalibrated before use:", teacher_record["recalibrated_before_use"])

In [ ]:
# Stage 6 - recovery runs: dense and Fisher-pruned x five seeds, eight units, dual-mode audit every unit, per-batch trace, remedy on collapses
EPOCHS_CSV = OUT / "recovery_epochs.csv"
rows = pd.read_csv(EPOCHS_CSV).to_dict("records") if EPOCHS_CSV.exists() else []
done = {r["tag"] for r in rows}


def fisher_student():
    rm = pd.read_csv(STRUCTURE_CSV)
    pm = {str(l): sorted(g["channel_index"].astype(int).tolist()) for l, g in rm.groupby("module_path")}
    st, _ = prune_cnn1d_channels(TEACHER, pm, EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_W)   # returns a pruned deep copy
    return st.to(DEVICE)


def remedy(model, unit, tag):
    # called on a gap-rule collapse: save the snapshot, recalibrate a copy on the fixed slice, record the result, leave the run untouched
    snap = SNAP_DIR / f"{tag.replace('/', '_')}_u{unit}.pt"
    torch.save({"state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, "tag": tag, "unit": unit}, snap)
    m = copy.deepcopy(model); recalibrate_bn(m, CAL_LOADER); rc = audit(m, EX_X, EX_Y, taxonomy, "eval"); del m
    return {f"recal_{k}": v for k, v in rc.items()}


JOBS = [(f"{kind}/shallow/s{seed}", kind, seed) for kind in ("dense", "fisher") for seed in SEEDS]
for tag, kind, seed in JOBS:
    if tag in done:
        continue
    t0 = time.time(); torch.manual_seed(seed); np.random.seed(seed)
    model = copy.deepcopy(TEACHER).to(DEVICE) if kind == "dense" else fisher_student()
    loader, sub = make_subset_loader(TRAIN_DS, seed, SUBSET_FRACTION, BATCH)
    sub_max_z = float(blob["X_tr"][sub].abs().max())
    ep, tr = recover(model, loader, N_UNITS, CLASS_W, EX_X, EX_Y, taxonomy, tag, trace_every=TRACE_EVERY, gap=COLLAPSE_GAP, on_collapse=lambda m, u, tag=tag: remedy(m, u, tag))
    for r_ in ep:
        r_["kind"] = kind; r_["seed"] = seed; r_["subset_max_abs_z"] = sub_max_z; r_["n_params"] = int(sum(p.numel() for p in model.parameters()))
    rows.extend(ep); pd.DataFrame(rows).to_csv(EPOCHS_CSV, index=False)
    pd.DataFrame(tr).to_csv(TRACE_DIR / f"trace_{tag.replace('/', '_')}.csv", index=False)
    hits = [r_["unit"] for r_ in ep if r_["is_collapse_gap"]]
    print(f"{tag}: subset max |z| {sub_max_z:.0f} | collapse (gap rule) at {hits or 'none'} | max unit gap {max(r_['gap'] for r_ in ep):.4f} | "
          f"final eval b2a {ep[-1]['eval_b2a']:.4f} | {round(time.time() - t0)} s", flush=True)
    del model; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None
epochs = pd.DataFrame(rows)
print("unit evaluations:", len(epochs), "| gap-rule collapses:", int(epochs.is_collapse_gap.sum()), "| absolute-rule collapses:", int(epochs.is_collapse_abs.sum()))

In [ ]:
# Stage 7 - verdict: V1 (collapse or not), V2 (trace: jumps at the extreme batch, late landings), V3 (remedy on collapses)
epochs = pd.read_csv(EPOCHS_CSV)
assert len(epochs) == 2 * len(SEEDS) * N_UNITS, len(epochs)
an_rows = []
for tag in epochs.tag.unique():
    tr = pd.read_csv(TRACE_DIR / f"trace_{tag.replace('/', '_')}.csv")
    for a in trace_analysis(tr, late_window=LATE_WINDOW):
        a["tag"] = tag; an_rows.append(a)
an = pd.DataFrame(an_rows); an.to_csv(OUT / "trace_analysis.csv", index=False)
ref = pd.read_csv(NB35_TRACE); ref_an = pd.DataFrame(trace_analysis(ref, late_window=LATE_WINDOW))

V1 = bool(epochs.is_collapse_gap.any())
late = an[an.late_landing]
col = epochs[epochs.is_collapse_gap]
V3 = bool(((col.recal_b2a - col.batch_b2a).abs() <= 0.02).all()) if (V1 and "recal_b2a" in col.columns) else None


def _records(df):
    return [{k: (v.item() if isinstance(v, np.generic) else v) for k, v in r.items()} for r in df.to_dict("records")]
reading = "unclipped_iomt_collapses" if V1 else ("spike_too_small" if len(late) else "untested_by_order")
verdict = {
    "arm": "V_unclipped_iomt_recovery",
    "V1_any_gap_collapse": V1, "V3_remedy_within_0.02": V3, "reading": reading,
    "unit_evaluations": int(len(epochs)), "gap_rule_collapses": int(epochs.is_collapse_gap.sum()), "absolute_rule_collapses": int(epochs.is_collapse_abs.sum()),
    "collapse_units": _records(col[["tag", "unit", "eval_b2a", "batch_b2a", "gap", "bn_var_max"]]),
    "max_gap_any_unit": float(epochs.gap.max()), "max_eval_b2a_any_unit": float(epochs.eval_b2a.max()),
    "final_unit": _records(epochs[epochs.unit == N_UNITS][["tag", "eval_b2a", "batch_b2a", "eval_a2b", "eval_family_f1", "gap"]]),
    "trace": {"units_traced": int(len(an)), "late_landings": int(len(late)),
              "late_landing_units": _records(late[["tag", "unit", "max_abs_z", "max_abs_z_batch", "n_batches", "jump_at_max_abs_z_batch", "final_var"]]),
              "jump_rank_1_share": float((an.jump_rank == 1).mean()),
              "jump_at_extreme_batch_median": float(an.jump_at_max_abs_z_batch.median()), "jump_at_extreme_batch_max": float(an.jump_at_max_abs_z_batch.max()),
              "var_before_median": float(an.var_before.median()), "subset_max_abs_z_by_seed": {int(k): float(v) for k, v in epochs.groupby("seed").subset_max_abs_z.first().round(1).items()}},
    "ciciot2023_reference_nb35": {"jump_at_extreme_batch_min": float(ref_an.jump_at_max_abs_z_batch.min()), "jump_at_extreme_batch_max": float(ref_an.jump_at_max_abs_z_batch.max()),
                                  "var_before_median": float(ref_an.var_before.median()), "max_abs_z": float(ref_an.max_abs_z.max()), "late_landings": int(ref_an.late_landing.sum())},
    "teacher": json.loads(TAUD.read_text()), "prereg": PREREG,
}
(OUT / "V43_verdict.json").write_text(json.dumps(verdict, indent=2, default=lambda o: o.item() if isinstance(o, np.generic) else str(o)))
print(json.dumps({k: verdict[k] for k in ("V1_any_gap_collapse", "V3_remedy_within_0.02", "reading", "unit_evaluations", "gap_rule_collapses", "absolute_rule_collapses", "max_gap_any_unit")}, indent=1))
print("trace:", {k: v for k, v in verdict["trace"].items() if k != "late_landing_units"})
print("CICIoT2023 reference:", verdict["ciciot2023_reference_nb35"])
if V1:
    print("collapse units:"); print(col[["tag", "unit", "eval_b2a", "batch_b2a", "gap", "recal_b2a"]].round(4).to_string(index=False))

In [ ]:
# Stage 8 - figure: the per-batch trace of the run with the largest running-variance jump (or the first collapse), 600 dpi, paper names
matplotlib.rcParams.update({"font.family": "serif", "font.size": 7, "pdf.fonttype": 42})
pick = (col.tag.iloc[0] if V1 else an.sort_values("jump_at_max_abs_z_batch", ascending=False).tag.iloc[0])
tr = pd.read_csv(TRACE_DIR / f"trace_{pick.replace('/', '_')}.csv"); nb_ = int(tr.batch.max()) + 1; tr["g"] = (tr.unit - 1) * nb_ + tr.batch
fig, (a1, a2) = plt.subplots(2, 1, figsize=(7.0, 3.2), sharex=True, gridspec_kw={"height_ratios": [1.25, 1.0], "hspace": 0.25})
a1.plot(tr.g, tr.bn_var_max, color="#0072B2", lw=0.8)
ext = tr.loc[tr.groupby("unit").batch_max_abs_z.idxmax()]
a1.scatter(ext.g, ext.bn_var_max, marker="v", s=18, color="#D55E00", zorder=5, label="batch holding the unit's largest |z|")
ends = tr[tr.batch == nb_ - 1]
a1.scatter(ends.g, ends.bn_var_max, marker="s", s=14, facecolor="white", edgecolor="black", lw=0.7, zorder=6, label="saved end-of-unit checkpoint")
for u in range(1, N_UNITS + 1):
    for a in (a1, a2):
        a.axvline(u * nb_, color="#BBBBBB", lw=0.5, ls=":")
a1.set_ylabel("Max. BatchNorm\nrunning variance"); a1.legend(loc="upper left", frameon=False, fontsize=6, ncol=2)
ev = tr.dropna(subset=["eval_b2a"]); a2.plot(ev.g, ev.eval_b2a * 100, color="black", lw=0.8, marker="o", ms=1.8)
a2.axhline(10, color="#D55E00", lw=0.6, ls="--"); a2.set_ylabel("Benign flows\nescalated (%)")
a2.set_xlabel(f"Recovery batch ({N_UNITS} units of {nb_} batches on the unclipped 10% subset), run {pick}")
for a in (a1, a2):
    a.spines[["top", "right"]].set_visible(False); a.grid(axis="y", color="#E6E6E6", lw=0.5); a.set_axisbelow(True)
fig.savefig(OUT / "V43_trace.png", dpi=600, bbox_inches="tight"); fig.savefig(OUT / "V43_trace.pdf", bbox_inches="tight"); plt.show()
print("figure written for run", pick)

In [ ]:
# Stage 9 - commit the results (standard commit cell; the message is filled in from the verdict)
v = json.load(open(OUT / "V43_verdict.json"))
pre = subprocess.run(["git", "log", "-1", "--format=%h", "--", "results/saber/43_iomt_unclipped_recovery/V43_PREREGISTRATION.json"], capture_output=True, text=True, cwd=REPO).stdout.strip() or "see log"
t, trc, ref = v["teacher"], v["trace"], v["ciciot2023_reference_nb35"]
msg = (f"NB43 results (pre-registration {pre}): V1 {'HELD' if v['V1_any_gap_collapse'] else 'NOT HELD'} -> {v['reading']}. "
       f"Unclipped CIC-IoMT-2024, shallow dense + Fisher-pruned x 5 seeds x 8 units under the NB31/34/35 recipe: "
       f"{v['gap_rule_collapses']} gap-rule collapses in {v['unit_evaluations']} unit evaluations ({v['absolute_rule_collapses']} by the absolute rule); "
       f"largest unit gap {v['max_gap_any_unit']:.4f}, largest stored-statistics benign escalation {v['max_eval_b2a_any_unit']:.4f}. "
       f"Trace: {trc['late_landings']} late landings in {trc['units_traced']} units; running-variance jump at the extreme batch median {trc['jump_at_extreme_batch_median']:.3f}, "
       f"max {trc['jump_at_extreme_batch_max']:.3f} (CICIoT2023 NB35: {ref['jump_at_extreme_batch_min']:.2f} to {ref['jump_at_extreme_batch_max']:.2f} at |z| {ref['max_abs_z']:.0f}); "
       f"extreme batch is the largest jump in {100 * trc['jump_rank_1_share']:.0f}% of units. "
       f"Unclipped teacher: eval b2a {t['as_trained_eval']['b2a']:.4f}, a2b {t['as_trained_eval']['a2b']:.4f}, fine F1 {t['as_trained_eval']['fine_f1']:.3f}, "
       f"meets NB32 rule {t['meets_nb32_rule']}, recalibrated before use {t['recalibrated_before_use']}. "
       f"V3 {'HELD' if v['V3_remedy_within_0.02'] else ('NOT HELD' if v['V3_remedy_within_0.02'] is False else 'n/a (no collapse)')}.")
print(msg)
os.chdir(REPO)
r = subprocess.run(["python", "tools/commit_cell.py", msg], capture_output=True, text=True)
print(r.stdout); print(r.stderr)